# 05 — Data Freshness Check at Sign-Off Demo

Implements Chapter 5's `DataSnapshot` and `check_freshness` design: a lightweight fingerprint-and-compare check run at 'open for review' and again at 'submit approval', distinguishing **citation-verification-passed** (Chapter 3, internal coherence) from **currency-as-of-sign-off** (this chapter) as two genuinely independent concerns.

In [1]:
from dataclasses import dataclass
from datetime import datetime, timedelta
import hashlib

@dataclass
class DataSnapshot:
    customer_id: str
    kyc_profile_version_hash: str
    transaction_history_max_txn_id: str
    transaction_history_as_of: datetime
    generated_at: datetime

@dataclass
class FreshnessResult:
    stale: bool
    reason: str = None
    kyc_changed: bool = False
    new_transactions_since_generation: bool = False

def hash_kyc_profile(kyc: dict) -> str:
    canonical = str(sorted(kyc.items()))
    return hashlib.sha256(canonical.encode()).hexdigest()[:12]

print('DataSnapshot / FreshnessResult / hash_kyc_profile defined.')

DataSnapshot / FreshnessResult / hash_kyc_profile defined.


In [2]:
# Simulated 'live' data stores -- mutable, standing in for the KYC/transaction systems
_LIVE_KYC = {'CUST-4471': {'occupation': 'Import/export consultant', 'risk_rating': 'MEDIUM'}}
_LIVE_TXN_MAX_ID = {'CUST-4471': 'TXN-88240'}

def get_kyc_profile(customer_id):
    return _LIVE_KYC[customer_id]

def get_latest_transaction_id(customer_id):
    return _LIVE_TXN_MAX_ID[customer_id]

def check_freshness(snapshot: DataSnapshot) -> FreshnessResult:
    current_kyc_hash = hash_kyc_profile(get_kyc_profile(snapshot.customer_id))
    current_max_txn_id = get_latest_transaction_id(snapshot.customer_id)

    kyc_changed = current_kyc_hash != snapshot.kyc_profile_version_hash
    new_transactions = current_max_txn_id != snapshot.transaction_history_max_txn_id

    if kyc_changed or new_transactions:
        return FreshnessResult(
            stale=True,
            reason='Customer data has changed since this draft was generated -- regenerate before approving.',
            kyc_changed=kyc_changed,
            new_transactions_since_generation=new_transactions,
        )
    return FreshnessResult(stale=False)

print('check_freshness() defined.')

check_freshness() defined.


## Scenario 1 — narrative generated, no changes yet, review opens: fresh

In [3]:
generated_at = datetime.utcnow() - timedelta(hours=1)
snapshot = DataSnapshot(
    customer_id='CUST-4471',
    kyc_profile_version_hash=hash_kyc_profile(_LIVE_KYC['CUST-4471']),
    transaction_history_max_txn_id=_LIVE_TXN_MAX_ID['CUST-4471'],
    transaction_history_as_of=generated_at,
    generated_at=generated_at,
)

result = check_freshness(snapshot)
print(f"Freshness check at 'open for review': stale={result.stale}")
assert result.stale is False
print('PASS: no data has changed since generation -- narrative is current.')

Freshness check at 'open for review': stale=False
PASS: no data has changed since generation -- narrative is current.


C:\Users\abhis\AppData\Local\Temp\ipykernel_22824\2690157026.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  generated_at = datetime.utcnow() - timedelta(hours=1)


## Scenario 2 — a new transaction posts while the narrative sits in the review queue

In [4]:
# Time passes -- the officer's queue is backed up, and in the interim a NEW transaction posts
_LIVE_TXN_MAX_ID['CUST-4471'] = 'TXN-91055'  # a transaction the narrative never saw

result_at_submit = check_freshness(snapshot)
print(f"Freshness check at 'submit approval': stale={result_at_submit.stale}")
print(f"  reason: {result_at_submit.reason}")
print(f"  kyc_changed={result_at_submit.kyc_changed}, new_transactions_since_generation={result_at_submit.new_transactions_since_generation}")

assert result_at_submit.stale is True
assert result_at_submit.new_transactions_since_generation is True
assert result_at_submit.kyc_changed is False
print('\nPASS: the check correctly flags staleness at submit-time even though it passed at open-time --')
print('      exactly the "checked at two moments, not continuously" design from Chapter 5 Part 3.')

Freshness check at 'submit approval': stale=True
  reason: Customer data has changed since this draft was generated -- regenerate before approving.
  kyc_changed=False, new_transactions_since_generation=True

PASS: the check correctly flags staleness at submit-time even though it passed at open-time --
      exactly the "checked at two moments, not continuously" design from Chapter 5 Part 3.


## The two independent axes, demonstrated together

A narrative can pass citation verification (Notebook 03) while failing freshness (this notebook) — proving Chapter 5's core claim that these are genuinely orthogonal checks, not the same thing measured twice.

In [5]:
# Reuse the clean, fully-grounded narrative's citation-check result conceptually:
citation_check_passed = True   # every citation resolves against ITS OWN generation-time snapshot
freshness_check_passed = not result_at_submit.stale  # False -- data has since changed

print(f"Citation verification passed: {citation_check_passed}")
print(f"Freshness check passed:       {freshness_check_passed}")
print()
assert citation_check_passed and not freshness_check_passed, (
    'This is exactly the failure mode Chapter 5 Part 2 warns against: a narrative can be '
    'perfectly grounded in its own snapshot while that snapshot is now stale.'
)
print('PASS: citation verification and freshness are independent -- one can pass while the other fails,')
print('confirming these are two different questions, not the same check run twice.')
print()
print('Per Chapter 5 Part 3: this narrative would now surface a hard, overridable interstitial')
print('to the reviewing officer -- "customer data has changed since this draft was generated" --')
print('rather than a quiet badge easy to miss under time pressure.')

Citation verification passed: True
Freshness check passed:       False

PASS: citation verification and freshness are independent -- one can pass while the other fails,
confirming these are two different questions, not the same check run twice.

Per Chapter 5 Part 3: this narrative would now surface a hard, overridable interstitial
to the reviewing officer -- "customer data has changed since this draft was generated" --
rather than a quiet badge easy to miss under time pressure.
